In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


# =========================
# Config
# =========================
ROOT = Path(r"D:\mmwave-heart-rate-monitoring-demo\results\Heart_estimation_Cepstrum")

FILES = [
    "AWR_steady.csv",
    "AWR_unsteady.csv",
    # "IWR_steady.csv",
    # "IWR_unsteady.csv",
]

METHOD_ORDER = [
    "raw",
    "AMP",
    "BP",
    "Median",
    "Wavelet",
    "AMP->BP",
    "AMP->Median",
    "AMP->Wavelet",
    "BP->AMP",
    "BP->Median",
    "BP->Wavelet",
    "Median->AMP",
    "Median->BP",
    "Median->Wavelet",
    "Wavelet->AMP",
    "Wavelet->BP",
    "Wavelet->Median",
]

METHOD_TO_COLUMN = {
    "raw": "raw_CEP_HR",
    "AMP": "AMP_CEP_HR",
    "BP": "BP_CEP_HR",
    "Median": "Median_CEP_HR",
    "Wavelet": "Wavelet_CEP_HR",
    "AMP->BP": "AMP->BP_CEP_HR",
    "AMP->Median": "AMP->Median_CEP_HR",
    "AMP->Wavelet": "AMP->Wavelet_CEP_HR",
    "BP->AMP": "BP->AMP_CEP_HR",
    "BP->Median": "BP->Median_CEP_HR",
    "BP->Wavelet": "BP->Wavelet_CEP_HR",
    "Median->AMP": "Median->AMP_CEP_HR",
    "Median->BP": "Median->BP_CEP_HR",
    "Median->Wavelet": "Median->Wavelet_CEP_HR",
    "Wavelet->AMP": "Wavelet->AMP_CEP_HR",
    "Wavelet->BP": "Wavelet->BP_CEP_HR",
    "Wavelet->Median": "Wavelet->Median_CEP_HR",
}


# =========================
# Metrics
# =========================
def compute_metrics(ecg, mmw):
    error = mmw - ecg
    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error ** 2))
    bias = np.mean(error)
    std = np.std(error, ddof=1) if len(error) > 1 else 0.0
    ci = 1.96 * std
    return mae, rmse, bias, std, ci


# =========================
# Analysis
# =========================
for file in FILES:
    path = ROOT / file

    if not path.exists():
        print(f"找不到檔案: {path}")
        continue

    df = pd.read_csv(path)

    # 保險：把欄位前後空白去掉
    df.columns = df.columns.str.strip()

    if "ECG_HR" not in df.columns:
        print(f"{file} 缺少 ECG_HR 欄位")
        continue

    ecg = pd.to_numeric(df["ECG_HR"], errors="coerce")
    rows = []

    for method in METHOD_ORDER:
        col = METHOD_TO_COLUMN[method]

        if col not in df.columns:
            print(f"找不到欄位: {col}")
            rows.append({
                "Method": method,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Std": np.nan,
                "95% CI": np.nan
            })
            continue

        mmw = pd.to_numeric(df[col], errors="coerce")
        valid = np.isfinite(mmw) & np.isfinite(ecg)

        if valid.sum() == 0:
            rows.append({
                "Method": method,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Std": np.nan,
                "95% CI": np.nan
            })
            continue

        mae, rmse, bias, std, ci = compute_metrics(
            ecg[valid].to_numpy(),
            mmw[valid].to_numpy()
        )

        rows.append({
            "Method": method,
            "MAE": round(mae, 3),
            "RMSE": round(rmse, 3),
            "Bias": round(bias, 3),
            "Std": round(std, 3),
            "95% CI": round(ci, 3)
        })

    result = pd.DataFrame(rows)

    print("\n" + "=" * 60)
    print(file)
    print("=" * 60)
    print(result.to_string(index=False))

    # # 存檔
    # out_path = ROOT / f"{Path(file).stem}_stats.csv"
    # result.to_csv(out_path, index=False, encoding="utf-8-sig")
    # print(f"\n已儲存: {out_path}")


AWR_steady.csv
         Method    MAE   RMSE    Bias    Std  95% CI
            raw  7.064  9.079  -5.376  7.390  14.484
            AMP  5.824  7.863  -3.576  7.073  13.864
             BP  6.264  7.913  -4.576  6.522  12.782
         Median 32.897 36.629  32.777 16.517  32.373
        Wavelet 40.717 43.117  40.077 16.066  31.490
        AMP->BP  5.764  7.754  -4.076  6.663  13.059
    AMP->Median 32.897 36.629  32.777 16.517  32.373
   AMP->Wavelet 44.827 47.927  44.827 17.130  33.574
        BP->AMP  4.824  6.198  -2.976  5.493  10.765
     BP->Median 33.077 34.776  33.077 10.846  21.259
    BP->Wavelet 38.467 39.598  37.827 11.829  23.185
    Median->AMP 32.427 36.220  32.427 16.301  31.949
     Median->BP 14.228 15.375 -14.148  6.080  11.916
Median->Wavelet 40.327 41.803  40.327 11.123  21.801
   Wavelet->AMP 39.027 41.826  39.027 15.198  29.788
    Wavelet->BP 32.297 47.337  20.723 42.991  84.263
Wavelet->Median 70.830 74.346  70.830 22.821  44.730

AWR_unsteady.csv
         Met